# Fault Tolerance in LangGraph

Real graphs run into flaky nodes, slow nodes, and infinite loops. This notebook covers the tools LangGraph gives you to handle those failures: retries, error handlers, timeouts, recursion limits, and node caching. Most examples use plain Python nodes so the failure behavior is easy to see without an LLM in the way.

In [1]:
import time
import asyncio
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy, CachePolicy, TimeoutPolicy, Command
from langgraph.errors import NodeError, NodeTimeoutError, GraphRecursionError
from langgraph.cache.memory import InMemoryCache

## Part 1: Retrying a Flaky Node

A node can fail because of a transient problem, like a dropped connection. `RetryPolicy` tells LangGraph to call the node again instead of failing the whole run. `retry_on` limits retries to specific exception types, so a real bug in your code still fails immediately instead of being retried.

In [2]:
class CounterState(TypedDict):
    n: int

attempts = {"count": 0}

def flaky_node(state: CounterState) -> dict:
    attempts["count"] += 1
    print(f"attempt {attempts['count']}")
    if attempts["count"] < 3:
        raise ConnectionError("simulated network blip")
    return {"n": state["n"] + 1}

retry_graph = (
    StateGraph(CounterState)
    .add_node(
        "flaky_node",
        flaky_node,
        retry_policy=RetryPolicy(max_attempts=5, initial_interval=0.01, retry_on=ConnectionError),
    )
    .add_edge(START, "flaky_node")
    .add_edge("flaky_node", END)
    .compile()
)

print(retry_graph.invoke({"n": 0}))

attempt 1


attempt 2


attempt 3
{'n': 1}


## Part 2: A Retry Policy on an LLM Node

A `retry_policy` works on any node, including one that calls an LLM. LLM calls can fail from rate limits or network errors, so wrapping the LLM call with a retry policy is a common production pattern. It looks the same as the plain Python case above, just with an LLM call inside the node:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

class ChatState(TypedDict):
    messages: list[BaseMessage]

def llm_node(state: ChatState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [response]}

llm_graph = (
    StateGraph(ChatState)
    .add_node("llm_node", llm_node, retry_policy=RetryPolicy(max_attempts=3))
    .add_edge(START, "llm_node")
    .add_edge("llm_node", END)
    .compile()
)

llm_graph.invoke({"messages": [HumanMessage(content="Name one benefit of retries, in one sentence.")]})
```

This code cell is not run in this notebook (no LLM calls are made here); everything else in this notebook is plain Python so you can see the retry/timeout/cache behavior directly.

## Part 3: Handling a Node That Keeps Failing

Retries only help with transient failures. If a node runs out of retries (or has none), `error_handler` runs instead of crashing the graph. The handler receives the state and a `NodeError` (with `.node` and `.error`), and returns a `Command` that updates state and picks the next node.

In [3]:
class TaskState(TypedDict):
    status: str

def boom_node(state: TaskState) -> dict:
    raise ValueError("task processor is down")

def handle_boom(state: TaskState, error: NodeError) -> Command:
    print(f"node '{error.node}' failed with: {error.error!r}")
    return Command(update={"status": "recovered"}, goto=END)

handler_graph = (
    StateGraph(TaskState)
    .add_node("boom_node", boom_node, error_handler=handle_boom)
    .add_edge(START, "boom_node")
    .compile()
)

print(handler_graph.invoke({"status": "pending"}))

node 'boom_node' failed with: ValueError('task processor is down')
{'status': 'recovered'}


## Part 4: Timing Out a Slow Node

A node can also hang instead of failing outright, for example a slow API call. `timeout=` cancels the node after a set number of seconds and raises `NodeTimeoutError`. This only works on **async** nodes, so the node must be `async def` and the graph must be run with `ainvoke`/`astream`. `TimeoutPolicy` gives more control than a plain number of seconds, such as a separate `idle_timeout`.

In [4]:
class SlowState(TypedDict):
    n: int

async def slow_node(state: SlowState) -> dict:
    await asyncio.sleep(1.0)
    return {"n": state["n"] + 1}

timeout_graph = (
    StateGraph(SlowState)
    .add_node("slow_node", slow_node, timeout=TimeoutPolicy(run_timeout=0.2))
    .add_edge(START, "slow_node")
    .add_edge("slow_node", END)
    .compile()
)

try:
    await timeout_graph.ainvoke({"n": 0})
except NodeTimeoutError as e:
    print(f"caught NodeTimeoutError: {e}")

caught NodeTimeoutError: Node 'slow_node' exceeded its run timeout of 0.200s (elapsed: 0.219s).


## Part 5: Stopping Infinite Loops

A graph can also fail by looping forever, for example a conditional edge that never routes to `END`. `recursion_limit`, passed in the run config, caps how many steps a single run can take. LangGraph raises `GraphRecursionError` once the cap is hit.

In [5]:
class LoopState(TypedDict):
    n: int

def loop_node(state: LoopState) -> dict:
    return {"n": state["n"] + 1}

loop_graph = (
    StateGraph(LoopState)
    .add_node("loop_node", loop_node)
    .add_edge(START, "loop_node")
    .add_edge("loop_node", "loop_node")
    .compile()
)

try:
    loop_graph.invoke({"n": 0}, {"recursion_limit": 5})
except GraphRecursionError as e:
    print(f"caught GraphRecursionError: {e}")

caught GraphRecursionError: Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT


## Part 6: Caching a Node's Result

Some nodes are slow but deterministic for the same input, like a lookup that hits an external system. `cache_policy=CachePolicy(ttl=...)` on the node, plus `compile(cache=InMemoryCache())` on the graph, lets LangGraph skip re-running the node when it sees the same input again within the TTL (time to live, in seconds).

In [6]:
class CacheState(TypedDict):
    n: int

calls = {"count": 0}

def slow_lookup(state: CacheState) -> dict:
    calls["count"] += 1
    time.sleep(0.3)
    return {"n": state["n"] + 1}

cache_graph = (
    StateGraph(CacheState)
    .add_node("slow_lookup", slow_lookup, cache_policy=CachePolicy(ttl=60))
    .add_edge(START, "slow_lookup")
    .add_edge("slow_lookup", END)
    .compile(cache=InMemoryCache())
)

start = time.time()
cache_graph.invoke({"n": 1})
mid = time.time()
cache_graph.invoke({"n": 1})
end = time.time()

print(f"first call: {mid - start:.2f}s, second call: {end - mid:.2f}s, node ran {calls['count']} time(s)")

first call: 0.30s, second call: 0.00s, node ran 1 time(s)


## Part 7: Setting Defaults for Every Node

Passing the same `retry_policy` to every `add_node` call gets repetitive once a graph has many nodes. `set_node_defaults` sets a default policy for every node in the graph. A policy passed directly to `add_node` still overrides the default for that node.

In [7]:
class DefaultState(TypedDict):
    n: int

default_attempts = {"count": 0}

def flaky_default(state: DefaultState) -> dict:
    default_attempts["count"] += 1
    if default_attempts["count"] < 2:
        raise ConnectionError("simulated blip")
    return {"n": state["n"] + 1}

defaults_graph = (
    StateGraph(DefaultState)
    .set_node_defaults(retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01, retry_on=ConnectionError))
    .add_node("flaky_default", flaky_default)
    .add_edge(START, "flaky_default")
    .add_edge("flaky_default", END)
    .compile()
)

print(defaults_graph.invoke({"n": 0}))
print(f"node ran {default_attempts['count']} time(s)")

{'n': 1}
node ran 2 time(s)


## Part 8: Surviving a Crash With a Checkpointer

Everything above handles failures *inside* a single run. A checkpointer handles a crash of the whole process. `09_Persistence` covers checkpointers for conversation memory; the same mechanism gives durable execution:

- LangGraph saves state after every super-step (every node completion), keyed by `thread_id`.
- If the process crashes mid-run, the last saved checkpoint is still on disk (or in whatever store the checkpointer uses).
- Calling `invoke(None, config)` (input `None`) with the same `thread_id` resumes from the last successful step. Nodes that already completed are not re-run. Passing the original input again would start a new run from `START` instead.

This is why combining a checkpointer with retries and timeouts matters: retries and timeouts handle a single node failing, and the checkpointer handles the whole process failing.

## Summary

- `RetryPolicy` on `add_node(..., retry_policy=...)` retries a node on specific exceptions.
- `error_handler` on `add_node(..., error_handler=...)` runs after retries are exhausted and can redirect the graph instead of crashing it.
- `timeout=` / `TimeoutPolicy` on `add_node` cancels a slow **async** node and raises `NodeTimeoutError`.
- `recursion_limit` in the run config stops infinite loops and raises `GraphRecursionError`.
- `cache_policy=CachePolicy(ttl=...)` on a node, with `compile(cache=InMemoryCache())`, skips re-running a node for a repeated input.
- `set_node_defaults(...)` applies a policy to every node in the graph at once.
- A checkpointer (see `09_Persistence`) gives durable execution: resuming a run after a crash from the last successful step.